# 210. Test-Time Compute：怎样在预算下自适应分配采样/搜索成本？

> **面试问题：怎样基于每题预期收益与动作成本分配 cheap/medium/expensive 推理预算，并评测质量、延迟、regret、公平与策略漂移？**

## 先给结论

这里的关键不是调用一个安全/训练/推理框架，而是定义输入、状态、不变量、失败分支和独立的判断 oracle。下方仅以受控小数据验证机制；真实服务仍需替换模型、密钥管理、访问控制、审计、红队评测和线上 SLO。

## 一手资料

- [Adaptive Test-Time Compute Allocation](https://arxiv.org/abs/2604.14853)
- [Self-Consistency](https://arxiv.org/abs/2203.11171)
- [Tree of Thoughts](https://arxiv.org/abs/2305.10601)

In [ ]:
notebook_contract = {"mode": "controlled-demo", "oracle": "assertions", "production": "security-and-versioning-required"}  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["mode"] == "controlled-demo"  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["oracle"] == "assertions"  # 执行本行的状态、计算或校验逻辑。
assert "versioning" in notebook_contract["production"]  # 执行本行的状态、计算或校验逻辑。
assert len(notebook_contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 问题拆解：test-time scaling 的难点是把预算花给谁

重复采样、搜索或长推理能提升部分样本的质量，但成本有限。不能对每个请求固定采样 N 次：系统应使用可观测特征或预测收益，选择 cheap/medium/expensive 动作，并在全局预算下评测总收益与公平性。


In [ ]:
tasks = [{"id": "easy", "value": {"cheap": 0.9, "medium": 0.91, "expensive": 0.92}}, {"id": "hard", "value": {"cheap": 0.3, "medium": 0.55, "expensive": 0.75}}]  # 执行本行的状态、计算或校验逻辑。
cost = {"cheap": 1, "medium": 3, "expensive": 6}  # 执行本行的状态、计算或校验逻辑。
assert len(tasks) == 2  # 执行本行的状态、计算或校验逻辑。
assert cost["cheap"] < cost["medium"] < cost["expensive"]  # 执行本行的状态、计算或校验逻辑。
assert tasks[1]["value"]["expensive"] > tasks[1]["value"]["cheap"]  # 执行本行的状态、计算或校验逻辑。


## 2. 局部决策：收益减去 lambda 加权成本

拉格朗日形式把全局平均预算转换为单样本效用 `accuracy - lambda * cost`。lambda 越大，动作越节省；这提供可解释 oracle，但真实收益必须来自离线校准或在线受控实验，不能凭模型自信声明。


In [ ]:
def choose_with_lambda(task, cost, penalty):  # 执行本行的状态、计算或校验逻辑。
    return max(task["value"], key=lambda action: task["value"][action] - penalty * cost[action])  # 执行本行的状态、计算或校验逻辑。
assert choose_with_lambda(tasks[0], cost, 0.0) == "expensive"  # 执行本行的状态、计算或校验逻辑。
assert choose_with_lambda(tasks[0], cost, 0.1) == "cheap"  # 执行本行的状态、计算或校验逻辑。
assert choose_with_lambda(tasks[1], cost, 0.0) == "expensive"  # 执行本行的状态、计算或校验逻辑。


## 3. 全局预算：用小动作空间穷举作为精确 oracle

相邻升级的性价比贪心会错过组合最优，例如先选 hard→medium 后可能再也无法升级到 expensive。教学代码穷举所有动作组合求精确解；生产可用受验证的优化/策略模型，但要与这个小 oracle 回归对照。


In [ ]:
from itertools import product  # 执行本行的状态、计算或校验逻辑。
def allocate(tasks, cost, budget):  # 执行本行的状态、计算或校验逻辑。
    best = None  # 执行本行的状态、计算或校验逻辑。
    for choices in product(("cheap", "medium", "expensive"), repeat=len(tasks)):  # 执行本行的状态、计算或校验逻辑。
        actions = {task["id"]: action for task, action in zip(tasks, choices)}  # 执行本行的状态、计算或校验逻辑。
        spent = sum(cost[action] for action in choices)  # 执行本行的状态、计算或校验逻辑。
        quality = sum(task["value"][actions[task["id"]]] for task in tasks)  # 执行本行的状态、计算或校验逻辑。
        if spent <= budget and (best is None or quality > best[0]):  # 执行本行的状态、计算或校验逻辑。
            best = (quality, spent, actions)  # 执行本行的状态、计算或校验逻辑。
    if best is None:  # 执行本行的状态、计算或校验逻辑。
        raise ValueError("预算不足以执行最便宜动作")  # 执行本行的状态、计算或校验逻辑。
    return best[2], best[1]  # 执行本行的状态、计算或校验逻辑。
allocation, spent = allocate(tasks, cost, 7)  # 执行本行的状态、计算或校验逻辑。
assert allocation["hard"] == "expensive"  # 执行本行的状态、计算或校验逻辑。
assert allocation["easy"] == "cheap"  # 执行本行的状态、计算或校验逻辑。
assert spent == 7  # 执行本行的状态、计算或校验逻辑。


## 4. 收益 oracle：用动作对应的验证成功概率汇总

生产中要用真实 verifier、人工标注或经校准的 reward model 衡量收益。这里用预先给定 value 表示验证集上的成功率，明确区分“预测/估计”与运行时实际是否正确。


In [ ]:
def expected_quality(tasks, allocation):  # 执行本行的状态、计算或校验逻辑。
    return sum(task["value"][allocation[task["id"]]] for task in tasks) / len(tasks)  # 执行本行的状态、计算或校验逻辑。
adaptive_quality = expected_quality(tasks, allocation)  # 执行本行的状态、计算或校验逻辑。
uniform_quality = expected_quality(tasks, {"easy": "medium", "hard": "medium"})  # 执行本行的状态、计算或校验逻辑。
assert adaptive_quality == (0.9 + 0.75) / 2  # 执行本行的状态、计算或校验逻辑。
assert uniform_quality == (0.91 + 0.55) / 2  # 执行本行的状态、计算或校验逻辑。
assert adaptive_quality > uniform_quality  # 执行本行的状态、计算或校验逻辑。


## 5. 准入：预算、截止时间和风险策略共同限制动作

昂贵动作可能超过用户 deadline、租户配额或安全策略。选择器不能只优化预期质量；必须先通过 admission gate，并在资源不足时降级到可解释的便宜策略，而不是无限重试。


In [ ]:
def admit(action, remaining_budget, deadline_ms, predicted_ms):  # 执行本行的状态、计算或校验逻辑。
    return cost[action] <= remaining_budget and predicted_ms[action] <= deadline_ms  # 执行本行的状态、计算或校验逻辑。
predicted_ms = {"cheap": 50, "medium": 150, "expensive": 400}  # 执行本行的状态、计算或校验逻辑。
assert admit("medium", 3, 200, predicted_ms)  # 执行本行的状态、计算或校验逻辑。
assert not admit("expensive", 7, 200, predicted_ms)  # 执行本行的状态、计算或校验逻辑。
assert not admit("medium", 2, 200, predicted_ms)  # 执行本行的状态、计算或校验逻辑。


## 6. 失败分支：收益预测错误会造成预算浪费

难度预测器可能把简单题判为昂贵、把困难题判为便宜。评估必须按预测分桶与真实收益分桶比较，监控 regret、超预算和群体差异；错误时应回退到稳定 baseline 并重新训练策略。


In [ ]:
import math  # 执行本行的状态、计算或校验逻辑。
def regret(oracle_value, selected_value):  # 执行本行的状态、计算或校验逻辑。
    return max(0.0, oracle_value - selected_value)  # 执行本行的状态、计算或校验逻辑。
assert math.isclose(regret(0.75, 0.55), 0.2)  # 执行本行的状态、计算或校验逻辑。
assert regret(0.55, 0.75) == 0.0  # 执行本行的状态、计算或校验逻辑。
assert regret(0.9, 0.9) == 0.0  # 执行本行的状态、计算或校验逻辑。


## 7. 评测：质量、成本、延迟与公平一起画曲线

只报平均 accuracy 会掩盖成本爆炸或某类请求被系统性降级。至少按预算点报告 expected/verified quality、平均/尾延迟、每个动作比例、超预算率和任务/租户切片。


In [ ]:
def allocation_metrics(tasks, allocation, cost):  # 执行本行的状态、计算或校验逻辑。
    return {"quality": expected_quality(tasks, allocation), "mean_cost": sum(cost[allocation[task["id"]]] for task in tasks) / len(tasks), "expensive_share": sum(allocation[task["id"]] == "expensive" for task in tasks) / len(tasks)}  # 执行本行的状态、计算或校验逻辑。
metrics = allocation_metrics(tasks, allocation, cost)  # 执行本行的状态、计算或校验逻辑。
assert metrics["quality"] == adaptive_quality  # 执行本行的状态、计算或校验逻辑。
assert metrics["mean_cost"] == 3.5  # 执行本行的状态、计算或校验逻辑。
assert metrics["expensive_share"] == 0.5  # 执行本行的状态、计算或校验逻辑。


## 8. 制品：策略依赖模型、收益预测、成本和预算版本

一次分配决策需要记录 base model、verifier/reward 版本、收益预测器、动作成本、预算、deadline 和最终 action。没有这些字段，线上质量变化无法归因，也不能重放或审计成本。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
import json  # 执行本行的状态、计算或校验逻辑。
artifact = {"model": "reasoner-v1", "value_model": "calibration-v1", "cost": cost, "budget": 7, "allocation": allocation}  # 执行本行的状态、计算或校验逻辑。
fingerprint = hashlib.sha256(json.dumps(artifact, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert artifact["budget"] == spent  # 执行本行的状态、计算或校验逻辑。
assert artifact["allocation"] == allocation  # 执行本行的状态、计算或校验逻辑。
assert len(fingerprint) == 64  # 执行本行的状态、计算或校验逻辑。


## 面试收束

完整回答应覆盖目标、显式数据结构、核心规则、边界失败、评测指标和版本制品。受控断言只验证实现不变量，不能被解读为真实模型质量、攻击鲁棒性、隐私合规或线上成本结论。
